# core

> Serve Python functions as MCP tools, and call MCP servers

In [ ]:
#| default_exp core

Serve Python functions as MCP tools, and call any MCP server's tools as Python functions.

## Serving tools

A tool is a docmented Python function, sync or async: fastcore's `get_schema` turns its docments into the MCP `inputSchema`, so there is no registration ceremony beyond passing functions to `MCPServer`. The server core is one stateless `dispatch` from message dict to reply dict — it speaks the `initialize`-era protocol current clients use, while keeping the shape the 2026-07-28 stateless revision standardized. `serve_stdio` serves it as newline-delimited JSON for host-launched local servers; `create_app` wraps it as a mountable ASGI app with one POST endpoint, and `serve_mcp` runs that under uvicorn.

## Auth

HTTP auth is a static bearer token checked by `auth_app`, raw ASGI middleware with a constant-time compare — no OAuth machinery. The policy lives in `serve_mcp`: a tool server is remote code execution, so a non-loopback bind refuses to start without a token (`$MCPMINI_TOKEN` or `token=`) unless `no_token=True` says auth lives elsewhere, e.g. a VPN.

## The command

`mcpmini tools.py` serves a file of docmented functions over stdio — the argv shape MCP host configs launch — and `--transport http` with the flags of `serve_mcp` covers remote deployment; `load_server` is the same file-to-server step from Python. The file's docstring becomes the server's `instructions`.

## Calling servers

`MCPClient.stdio(argv)` and `MCPClient.http(url)` drive the handshake and turn the server's `tools/list` into bound Python callables with real signatures, docs, and defaults (`mk_tool`, the mirror of `get_schema`). Bound tools return reply text and raise on `isError`; `call_tool` returns the raw result. The HTTP transport also handles what other servers send that ours doesn't: SSE response bodies and `Mcp-Session-Id` minting. Both directions are exercised against the official SDK's opposite half.

Every current MCP client speaks the 2025-11-25-era protocol: an `initialize` handshake, then plain JSON-RPC requests. The 2026-07-28 revision removes the handshake and makes every request self-describing, so the implementation here keeps all per-connection ceremony in the transports and out of the dispatch path: the server core is a stateless function from one message to one response, which is both the simplest shape today and the shape the protocol is moving to.

In [ ]:
#| export
import asyncio, httpx, importlib.util, inspect, json, secrets, sys, traceback
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.script import call_parse
from fastcore.funccall import get_schema, mk_tool
from starlette.applications import Starlette
from starlette.responses import JSONResponse, Response
from starlette.routing import Route
from mcpmini import __version__


In [ ]:
from fastcore.test import *
import shutil, socket, tempfile

## Messages

A message is a JSON-RPC dict, and a server turns each one into at most one reply: a `result` for success, an `error` with a numeric code for protocol failures, and nothing at all for notifications (messages without an `id`). These two builders are the only reply shapes in the protocol.

In [ ]:
#| export
def jresp(id, result): return dict(jsonrpc='2.0', id=id, result=result)
def jerr(id, code, message):
    "JSON-RPC error reply; MCP uses -32601 for unknown methods, -32602 for bad params, -32603 for handler failures"
    return dict(jsonrpc='2.0', id=id, error=dict(code=code, message=message))

## Tools from functions

An MCP tool definition is a name, a description, and a JSON schema for the arguments. A docmented Python function already carries all three, and fastcore's `get_schema` extracts them; MCP spells the schema key `inputSchema`. So in mcpmini, *a tool is just a docmented function*:

In [ ]:
def fahrenheit(
    celsius:float, # Temperature to convert
)->float:
    "Convert Celsius to Fahrenheit"
    return celsius*9/5+32

get_schema(fahrenheit, pname='inputSchema')

{'name': 'fahrenheit',
 'description': 'Convert Celsius to Fahrenheit\n\nReturns:\n- type: number',
 'inputSchema': {'type': 'object',
  'properties': {'celsius': {'description': 'Temperature to convert',
    'type': 'number'}},
  'required': ['celsius']}}

A tool's return value travels back as a list of content blocks. A string becomes one text block, any other value its `repr` (which an LLM reads more reliably than `str`, since nested strings keep their quotes), and a dict already carrying `content` passes through untouched, so a tool can produce images or multiple blocks when it needs to.

In [ ]:
#| export
def _content(res):
    "A `tools/call` result for return value `res`"
    if isinstance(res, dict) and 'content' in res: return res
    return dict(content=[dict(type='text', text=res if isinstance(res, str) else repr(res))], isError=False)

In [ ]:
test_eq(_content('hi')['content'], [dict(type='text', text='hi')])
test_eq(_content(212.0)['content'][0]['text'], '212.0')
passthru = dict(content=[dict(type='image', data='AAAA', mimeType='image/png')])
test_eq(_content(passthru), passthru)
_content(dict(a=1))

{'content': [{'type': 'text', 'text': "{'a': 1}"}], 'isError': False}

## The server

`MCPServer` is the whole server core: a name, a version, optional instructions for the client's model, and the tool table built from the functions' docments. Everything protocol-shaped lives in one `dispatch` method that maps a message dict to a reply dict, with no per-connection state anywhere — transports call it, tests call it directly.

In [ ]:
#| export
PROTO_VERSIONS = '2025-11-25','2025-06-18','2025-03-26'

class MCPServer:
    "An MCP tool server: `dispatch` maps one message dict to one reply dict"
    def __init__(self,
        name, # Server name reported to clients
        tools=(), # Functions (sync or async) exposed as tools, described by their docments
        version=__version__, # Server version reported to clients
        instructions=None, # Optional guidance forwarded to the client's model
    ):
        self.name,self.version,self.instructions = name,version,instructions
        self.specs = [get_schema(f, pname='inputSchema') for f in tools]
        self.funcs = {s['name']:f for s,f in zip(self.specs,tools)}

`initialize` echoes any protocol version we support and otherwise offers our newest, per the negotiation rules. A tool that raises reports its traceback *in-band* as an `isError` result — the spec distinguishes execution errors, which the model should see and react to, from protocol errors like an unknown tool name, which get JSON-RPC error replies.

In [ ]:
#| export
async def _acall(f, args):
    "A `tools/call` result from calling `f`, exceptions in-band as `isError` content"
    try:
        res = f(**args)
        if inspect.iscoroutine(res): res = await res
        return _content(res)
    except Exception: return dict(content=[dict(type='text', text=traceback.format_exc())], isError=True)

@patch
async def dispatch(self:MCPServer, msg):
    "The reply dict for one JSON-RPC message dict; None for notifications"
    id,meth,p = msg.get('id'),msg.get('method',''),msg.get('params',{})
    if 'id' not in msg: return None
    try:
        if meth=='initialize':
            pv = p.get('protocolVersion')
            r = dict(protocolVersion=pv if pv in PROTO_VERSIONS else PROTO_VERSIONS[0],
                capabilities=dict(tools={}), serverInfo=dict(name=self.name, version=self.version))
            if self.instructions: r['instructions'] = self.instructions
            return jresp(id, r)
        if meth=='tools/list': return jresp(id, dict(tools=self.specs))
        if meth=='tools/call':
            f = self.funcs.get(p['name'])
            if not f: return jerr(id, -32602, f"Unknown tool: {p['name']}")
            return jresp(id, await _acall(f, p.get('arguments',{})))
        if meth=='ping': return jresp(id, {})
        return jerr(id, -32601, f'Method not found: {meth}')
    except Exception as e: return jerr(id, -32603, f'{type(e).__name__}: {e}')

A server for the examples: the converter above, an async tool proving coroutines are awaited, and one that always raises. `initialize` echoes a version we know and substitutes our newest for one we don't (a 2026-era client probing an older server does exactly this dance):

In [ ]:
async def nap(
    ms:int=1, # How long to sleep, in milliseconds
)->str:
    "Sleep, then report back"
    await asyncio.sleep(ms/1000)
    return f'napped {ms}ms'

def crash()->str:
    "Always fails"
    return 1/0

srv = MCPServer('demo', [fahrenheit, nap, crash], instructions='Convert temperatures on request.')
init = await srv.dispatch(dict(jsonrpc='2.0', id=1, method='initialize', params=dict(protocolVersion='2025-06-18', capabilities={}, clientInfo=dict(name='t', version='0'))))
test_eq(init['result']['protocolVersion'], '2025-06-18')
old = await srv.dispatch(dict(jsonrpc='2.0', id=2, method='initialize', params=dict(protocolVersion='2024-11-05')))
test_eq(old['result']['protocolVersion'], PROTO_VERSIONS[0])
init['result']

{'protocolVersion': '2025-06-18',
 'capabilities': {'tools': {}},
 'serverInfo': {'name': 'demo', 'version': '0.0.1'},
 'instructions': 'Convert temperatures on request.'}

Notifications produce no reply at all — `notifications/initialized` is simply absorbed. The tool table lists every function, calls run sync or async tools alike, and each failure mode lands where a client expects it:

In [ ]:
test_eq(await srv.dispatch(dict(jsonrpc='2.0', method='notifications/initialized')), None)
test_eq([t['name'] for t in (await srv.dispatch(dict(jsonrpc='2.0', id=3, method='tools/list')))['result']['tools']], ['fahrenheit','nap','crash'])
call = lambda name,**kw: srv.dispatch(dict(jsonrpc='2.0', id=9, method='tools/call', params=dict(name=name, arguments=kw)))
test_eq((await call('fahrenheit', celsius=100))['result']['content'][0]['text'], '212.0')
test_eq((await call('nap', ms=2))['result']['content'][0]['text'], 'napped 2ms')
boom = (await call('crash'))['result']
assert boom['isError'] and 'ZeroDivisionError' in boom['content'][0]['text']
test_eq((await call('flux'))['error']['code'], -32602)
test_eq((await srv.dispatch(dict(jsonrpc='2.0', id=4, method='sampling/createMessage')))['error']['code'], -32601)
test_eq((await srv.dispatch(dict(jsonrpc='2.0', id=5, method='ping')))['result'], {})
await call('nap', ms='oops')

{'jsonrpc': '2.0',
 'id': 9,
 'result': {'content': [{'type': 'text',
    'text': 'Traceback (most recent call last):\n  File "<ipython-input-1-a09c7e0a21c6>", line 6, in _acall\n    if inspect.iscoroutine(res): res = await res\n                                       ^^^^^^^^^\n  File "<ipython-input-1-c40849cb62ff>", line 5, in nap\n    await asyncio.sleep(ms/1000)\n                        ~~^~~~~\nTypeError: unsupported operand type(s) for /: \'str\' and \'int\'\n'}],
  'isError': True}}

## stdio

The stdio transport is newline-delimited JSON: one message per line in, one reply per line out, until EOF. An unparseable line gets the JSON-RPC parse error with a null id, since there's no id to echo. `serve_stdio` works on any asyncio stream pair, with real stdin/stdout as the default — which is exactly how an MCP host launches a local server.

In [ ]:
#| export
async def _stdio_streams():
    loop = asyncio.get_running_loop()
    reader = asyncio.StreamReader()
    await loop.connect_read_pipe(lambda: asyncio.StreamReaderProtocol(reader), sys.stdin)
    tr,pr = await loop.connect_write_pipe(asyncio.streams.FlowControlMixin, sys.stdout)
    return reader,asyncio.StreamWriter(tr, pr, None, loop)

async def serve_stdio(
    srv, # `MCPServer` to serve
    reader=None, # asyncio stream to read messages from; stdin if None
    writer=None, # asyncio stream to write replies to; stdout if None
):
    "Serve `srv` over newline-delimited JSON-RPC until EOF, dispatching messages concurrently"
    if reader is None: reader,writer = await _stdio_streams()
    tasks,wlock = set(),asyncio.Lock()
    async def _one(msg):
        resp = jerr(None, -32700, 'Parse error') if msg is None else await srv.dispatch(msg)
        if resp is not None:
            async with wlock:
                writer.write(json.dumps(resp).encode()+b'\n')
                await writer.drain()
    while line := await reader.readline():
        if not line.strip(): continue
        try: msg = json.loads(line)
        except Exception: msg = None
        t = asyncio.create_task(_one(msg))
        tasks.add(t)
        t.add_done_callback(tasks.discard)
    if tasks: await asyncio.gather(*tasks, return_exceptions=True)

A socketpair gives real asyncio streams for exercising the loop without a subprocess. One side runs the server; the other plays client, sees replies for requests and silence for notifications, and closing the connection ends the serve task:

In [ ]:
a,b = socket.socketpair()
sr,sw = await asyncio.open_connection(sock=a)
cr,cw = await asyncio.open_connection(sock=b)
task = asyncio.create_task(serve_stdio(srv, sr, sw))
async def ask(msg):
    cw.write(json.dumps(msg).encode()+b'\n')
    await cw.drain()
    return json.loads(await cr.readline())
r = await ask(dict(jsonrpc='2.0', id=1, method='tools/call', params=dict(name='fahrenheit', arguments=dict(celsius=0))))
test_eq(r['result']['content'][0]['text'], '32.0')
cw.write(b'not json\n')
test_eq((await ask(dict(jsonrpc='2.0', id=2, method='ping')))['error']['code'], -32700)
test_eq((await cr.readline()), json.dumps(jresp(2, {})).encode()+b'\n')
cw.close()
await task
r

{'jsonrpc': '2.0',
 'id': 1,
 'result': {'content': [{'type': 'text', 'text': '32.0'}], 'isError': False}}

Dispatch is concurrent: a slow request does not block the ones behind it — which is what lets an `interrupt` tool call preempt a long-running `execute` on the same stdio connection:

In [ ]:
a2,b2 = socket.socketpair()
sr2,sw2 = await asyncio.open_connection(sock=a2)
cr2,cw2 = await asyncio.open_connection(sock=b2)
task2 = asyncio.create_task(serve_stdio(srv, sr2, sw2))
for m in (dict(jsonrpc='2.0', id=1, method='tools/call', params=dict(name='nap', arguments=dict(ms=300))),
    dict(jsonrpc='2.0', id=2, method='tools/call', params=dict(name='nap', arguments=dict(ms=1)))):
    cw2.write(json.dumps(m).encode()+b'\n')
replies = [json.loads(await cr2.readline()) for _ in range(2)]
test_eq([r['id'] for r in replies], [2, 1])
cw2.close()
await task2

## Streamable HTTP

The streamable HTTP transport is one POST endpoint. A server may answer any request with either a JSON body or an SSE stream; this one always answers plainly, which the spec permits and every client must accept (streaming earns its place when there are progress notifications to carry, which tool servers this simple don't have). Two deliberate absences, both matching where the protocol went in 2026-07-28: no `Mcp-Session-Id` is ever minted (one arriving from an older client is ignored), and GET gets `405` rather than a server-push stream.

Auth is a bearer token checked in raw ASGI middleware with a constant-time compare. Middleware rather than the route handler, so it guards everything mounted beneath it and composes with any ASGI host; raw ASGI rather than framework auth machinery, because a static personal token needs no OAuth resource-server ceremony.

In [ ]:
#| export
def auth_app(
    app, # ASGI app to guard
    token, # The bearer token every request must carry
):
    "Wrap `app` to reject requests whose `Authorization` bearer token doesn't match `token`"
    async def _f(scope, receive, send):
        if scope['type']=='http':
            hdr = dict(scope.get('headers') or ()).get(b'authorization', b'').decode()
            if not (hdr.startswith('Bearer ') and secrets.compare_digest(hdr[7:], token)):
                await send(dict(type='http.response.start', status=401, headers=[(b'www-authenticate', b'Bearer')]))
                return await send(dict(type='http.response.body', body=b'unauthorized'))
        await app(scope, receive, send)
    return _f

def create_app(
    srv, # `MCPServer` to expose
    token=None, # Bearer token required on every request; open if None
    path='/mcp', # Endpoint path
):
    "ASGI app serving `srv` over streamable HTTP"
    async def _post(req):
        resp = await srv.dispatch(await req.json())
        return JSONResponse(resp) if resp is not None else Response(status_code=202)
    app = Starlette(routes=[Route(path, _post, methods=['POST'])])
    return auth_app(app, token) if token else app

httpx's `ASGITransport` exercises the app with no socket involved. The guarded app refuses a missing or wrong token before any JSON is parsed, answers the right one, `405`s GET, and `202`s notifications:

In [ ]:
app = create_app(srv, token='sesame')
async def hpost(msg, token=None, meth='POST'):
    hdrs = {'Authorization': f'Bearer {token}'} if token else {}
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app), base_url='http://t') as c:
        return await c.request(meth, '/mcp', json=msg, headers=hdrs)
ping = dict(jsonrpc='2.0', id=1, method='ping')
test_eq((await hpost(ping)).status_code, 401)
test_eq((await hpost(ping, 'wrong')).status_code, 401)
test_eq((await hpost(ping)).headers['www-authenticate'], 'Bearer')
test_eq((await hpost(ping, 'sesame')).json()['result'], {})
test_eq((await hpost(ping, 'sesame', meth='GET')).status_code, 405)
test_eq((await hpost(dict(jsonrpc='2.0', method='notifications/initialized'), 'sesame')).status_code, 202)
(await hpost(dict(jsonrpc='2.0', id=2, method='tools/call', params=dict(name='fahrenheit', arguments=dict(celsius=37))), 'sesame')).json()

{'jsonrpc': '2.0',
 'id': 2,
 'result': {'content': [{'type': 'text', 'text': '98.6'}], 'isError': False}}

## Serving

`serve_mcp` is the deployment entry point, and it owns the auth policy: a tool server is remote code execution, so binding a non-loopback interface without a token refuses to start unless `no_token=True` says auth lives elsewhere (a VPN, a fronting proxy). The policy sits here rather than in `create_app` so that library users composing their own apps aren't second-guessed, while anyone *serving* gets the safe default — Jupyter's tokenless-by-default years are the cautionary tale.

In [ ]:
#| export
def _loopback(host): return host in ('127.0.0.1','localhost','::1')

@delegates(create_app)
async def serve_mcp(
    srv, # `MCPServer` to serve
    host='127.0.0.1', # Interface to bind
    port=8000, # Port to bind
    token=None, # Bearer token; `$MCPMINI_TOKEN` if None
    no_token=False, # Serve a non-loopback interface openly, when auth lives elsewhere (e.g. a VPN)
    **kwargs
):
    "Serve `srv` over streamable HTTP; non-loopback binds require a token unless `no_token`"
    import uvicorn
    token = token or os.environ.get('MCPMINI_TOKEN')
    if not (_loopback(host) or token or no_token): raise ValueError(f'binding {host} needs a token: pass `token`, set $MCPMINI_TOKEN, or say `no_token=True`')
    app = create_app(srv, token, **kwargs)
    await uvicorn.Server(uvicorn.Config(app, host=host, port=port, log_level='warning')).serve()

The refusal is immediate — nothing binds. A loopback bind without a token is the ordinary local case and serves happily; we start one on a spare port as the live server for the client examples below:

In [ ]:
with expect_fail(ValueError): await serve_mcp(srv, host='0.0.0.0')

def free_port():
    with socket.socket() as s:
        s.bind(('127.0.0.1', 0))
        return s.getsockname()[1]

port = free_port()
server_task = asyncio.create_task(serve_mcp(srv, port=port, token='sesame'))
url = f'http://127.0.0.1:{port}/mcp'
for _ in range(100):
    try:
        with socket.create_connection(('127.0.0.1', port), timeout=0.5): break
    except OSError: await asyncio.sleep(0.05)
url

'http://127.0.0.1:53517/mcp'

## The client

The client faces a wider world than the server: other people's servers *do* answer POSTs with SSE bodies, mint session ids, and negotiate versions. So the HTTP transport accepts both response shapes, echoes any `Mcp-Session-Id` it is given, and stamps the negotiated `MCP-Protocol-Version` on every request after `initialize`. An SSE body is `data:` lines grouped into blank-line-separated events:

In [ ]:
#| export
def sse_data(text):
    "JSON messages from the `data:` lines of SSE body `text`"
    res = []
    for ev in text.split('\n\n'):
        data = '\n'.join(l[5:].lstrip() for l in ev.splitlines() if l.startswith('data:'))
        if data: res.append(json.loads(data))
    return res

In [ ]:
test_eq(sse_data('event: message\ndata: {"a": 1}\n\n: keepalive\n\ndata: {"b":\ndata:  2}\n\n'), [dict(a=1), dict(b=2)])

Both transports expose one operation: `send` a message dict, get the matching reply dict back (or `None` for a notification). The stdio transport owns a subprocess and reads replies off its stdout, skipping any message that isn't the awaited reply (an old-era server may interleave notifications); the HTTP transport is a POST per message.

In [ ]:
#| export
class StdioTransport:
    "JSON-RPC to a subprocess over its stdin/stdout"
    def __init__(self,
        argv, # Server command line, e.g. `['mymcp', '--flag']`
        env=None, # Environment for the subprocess; inherited if None
        cwd=None, # Working directory for the subprocess; inherited if None
    ):
        self.argv,self.env,self.cwd,self.p,self.proto = argv,env,cwd,None,None
    async def start(self):
        self.p = await asyncio.create_subprocess_exec(*self.argv, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE,
            env=self.env, cwd=self.cwd)
    async def send(self, msg):
        self.p.stdin.write(json.dumps(msg).encode()+b'\n')
        await self.p.stdin.drain()
        if 'id' not in msg: return None
        while line := await self.p.stdout.readline():
            resp = json.loads(line)
            if resp.get('id')==msg['id']: return resp
        raise RuntimeError(f'server exited awaiting reply {msg["id"]}')
    async def aclose(self):
        if not self.p: return
        self.p.stdin.close()
        await self.p.wait()

class HTTPTransport:
    "JSON-RPC to a streamable HTTP endpoint, one POST per message"
    def __init__(self,
        url, # The MCP endpoint, e.g. 'http://127.0.0.1:8000/mcp'
        token=None, # Bearer token to send with every request
        http_client=None, # An `httpx.AsyncClient` to use, e.g. over `httpx.ASGITransport`; a fresh one if None
    ):
        self.headers = {'Accept': 'application/json, text/event-stream'}
        if token: self.headers['Authorization'] = f'Bearer {token}'
        self.url,self.client,self.sess,self.proto = url,http_client,None,None
    async def start(self):
        if self.client is None: self.client = httpx.AsyncClient()
    async def send(self, msg):
        h = dict(self.headers)
        if self.sess: h['Mcp-Session-Id'] = self.sess
        if self.proto: h['MCP-Protocol-Version'] = self.proto
        r = await self.client.post(self.url, json=msg, headers=h)
        r.raise_for_status()
        if sid := r.headers.get('mcp-session-id'): self.sess = sid
        if 'id' not in msg: return None
        if r.headers.get('content-type','').startswith('text/event-stream'):
            return first(m for m in sse_data(r.text) if m.get('id')==msg['id'])
        return r.json()
    async def aclose(self): await self.client.aclose()

`MCPClient` drives either transport through the handshake, then turns the server's `tools/list` into bound Python callables via fastcore's `mk_tool` — the mirror of `get_schema` on the server side, so a docmented function served at one end comes back out the other with its signature, docs, and defaults intact. Bound tools return the reply's text and raise on `isError`; `call_tool` gives the raw result dict when the blocks themselves matter.

In [ ]:
#| export
class MCPClient:
    "Call an MCP server's tools as Python functions"
    def __init__(self,
        tr, # A transport: `StdioTransport`, `HTTPTransport`, or compatible
    ):
        self.tr,self.tools,self._id = tr,AttrDict(),0
    @classmethod
    @delegates(StdioTransport)
    def stdio(cls, argv, **kwargs): return cls(StdioTransport(argv, **kwargs))
    @classmethod
    @delegates(HTTPTransport)
    def http(cls, url, **kwargs): return cls(HTTPTransport(url, **kwargs))
    async def rpc(self, method, **params):
        "One request round trip, returning its `result`; raises on an error reply"
        self._id += 1
        resp = await self.tr.send(dict(jsonrpc='2.0', id=self._id, method=method, params=params))
        if 'error' in resp: raise RuntimeError(f"{resp['error']['code']}: {resp['error']['message']}")
        return resp['result']
    async def start(self):
        "Handshake, then bind the server's tools; returns `self`"
        await self.tr.start()
        self.info = await self.rpc('initialize', protocolVersion=PROTO_VERSIONS[0], capabilities={},
            clientInfo=dict(name='mcpmini', version=__version__))
        self.tr.proto = self.info['protocolVersion']
        await self.tr.send(dict(jsonrpc='2.0', method='notifications/initialized'))
        for t in (await self.rpc('tools/list'))['tools']: self.tools[t['name']] = mk_tool(self.call_text, dict2obj(t))
        return self
    async def call_tool(self, name, **kw):
        "The raw `tools/call` result dict"
        return await self.rpc('tools/call', name=name, arguments=kw)
    async def call_text(self, name, **kw):
        "The text of a `tools/call` reply; raises if the tool errored"
        res = await self.call_tool(name, **kw)
        txt = '\n'.join(c['text'] for c in res.get('content',[]) if c.get('type')=='text')
        if res.get('isError'): raise RuntimeError(txt)
        return txt
    async def __aenter__(self): return await self.start()
    async def __aexit__(self, *args): await self.tr.aclose()

Against the live server started above: the handshake carries the instructions across, the rebuilt tool re-derives the exact schema it was served from (descriptions included, carried as `Annotated` metadata on its signature), and the bound tool wears `fahrenheit`'s real signature and docs, calling it round-trips, and a crashing tool surfaces as a Python exception carrying the server-side traceback:

In [ ]:
async with MCPClient.http(url, token='sesame') as c:
    test_eq(c.info['serverInfo']['name'], 'demo')
    test_eq(c.info['instructions'], 'Convert temperatures on request.')
    test_eq(list(inspect.signature(c.tools.fahrenheit).parameters), ['celsius'])
    test_eq(get_schema(c.tools.fahrenheit, pname='inputSchema')['inputSchema'], srv.specs[0]['inputSchema'])
    test_eq(await c.tools.fahrenheit(celsius=100), '212.0')
    test_eq(await c.tools.nap(), 'napped 1ms')
    with expect_fail(RuntimeError, contains='ZeroDivisionError'): await c.tools.crash()
    with expect_fail(RuntimeError, contains='-32602'): await c.call_tool('flux')
    doc_line = c.tools.fahrenheit.__doc__
doc_line

'Convert Celsius to Fahrenheit\n\nReturns:\n- type: number'

The same client over stdio, launching a small server file as a subprocess — the exported module serving for real, the way an MCP host launches a local server (the file recreates its tool, since a subprocess shares no state with this notebook):

In [ ]:
serverfile = Path(tempfile.mkdtemp())/'demo_stdio.py'
serverfile.write_text('''import asyncio
from mcpmini.core import MCPServer, serve_stdio

def fahrenheit(
    celsius:float, # Temperature to convert
)->float:
    "Convert Celsius to Fahrenheit"
    return celsius*9/5+32

asyncio.run(serve_stdio(MCPServer('demo-stdio', [fahrenheit])))
''')
async with MCPClient.stdio([sys.executable, str(serverfile)]) as c:
    test_eq(c.info['serverInfo']['name'], 'demo-stdio')
    res = await c.tools.fahrenheit(celsius=-40)
res


'-40.0'

## Interop

Talking to ourselves proves consistency, not correctness — for that, each side runs against the official SDK's other half. First the SDK client against our live server: a full session through its own machinery, list and call included. (`streams[:2]` because the SDK's v2 line drops the session-id callback from the tuple.)

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client, create_mcp_http_client

In [ ]:
async with create_mcp_http_client(headers={'Authorization': 'Bearer sesame'}) as hc, \
        streamable_http_client(url, http_client=hc) as streams, \
        ClientSession(*streams[:2]) as s:
    sdk_init = await s.initialize()
    test_eq({t.name for t in (await s.list_tools()).tools}, {'fahrenheit','nap','crash'})
    sdk_res = await s.call_tool('fahrenheit', dict(celsius=100))
test_eq(sdk_res.content[0].text, '212.0')
sdk_init.serverInfo

Implementation(name='demo', title=None, version='0.0.1', websiteUrl=None, icons=None)

And our client against the SDK's server, which stresses exactly the paths our own server never triggers: FastMCP answers POSTs with SSE bodies and mints an `Mcp-Session-Id` the client must echo. A successful round trip here means both of those client behaviors are real, not just unit-tested:

In [ ]:
import logging, uvicorn
from mcp.server.fastmcp import FastMCP


In [ ]:
logging.getLogger('mcp').setLevel(logging.WARNING)
fm = FastMCP('sdkdemo')
@fm.tool()
def double(x: int) -> int:
    "Twice `x`"
    return x*2

port2 = free_port()
sdk_task = asyncio.create_task(uvicorn.Server(uvicorn.Config(fm.streamable_http_app(), host='127.0.0.1', port=port2, log_level='warning')).serve())
for _ in range(100):
    try:
        with socket.create_connection(('127.0.0.1', port2), timeout=0.5): break
    except OSError: await asyncio.sleep(0.05)
async with MCPClient.http(f'http://127.0.0.1:{port2}/mcp') as c:
    test_eq(c.info['serverInfo']['name'], 'sdkdemo')
    assert c.tr.sess, 'SDK server should have minted a session id'
    dbl = await c.tools.double(x=21)
dbl

'42'

## The mcpmini command

MCP hosts launch stdio servers from an argv line, so the library needs a command: `mcpmini tools.py` serves a file's tools with no wrapper script. The unit is a Python file of docmented functions — its docstring becomes the server's `instructions`, its `__all__` (or, failing that, its public defined-here functions) picks the tools, and the file stem names the server.

In [ ]:
#| export
def _load_mod(path):
    spec = importlib.util.spec_from_file_location(Path(path).stem, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def load_server(
    path, # Python file whose public docmented functions become the tools
):
    "An `MCPServer` for the file at `path`: named after it, instructed by its docstring, serving its functions"
    mod = _load_mod(path)
    names = getattr(mod, '__all__', None)
    fs = [getattr(mod,n) for n in names] if names else [v for v in vars(mod).values() if inspect.isfunction(v) and v.__module__==mod.__name__]
    return MCPServer(mod.__name__, [f for f in fs if not f.__name__.startswith('_')], instructions=mod.__doc__)

A tools file for the examples: one public function, one private, one import — only the first should serve. Helpers imported *into* a file don't become tools, which is what makes `from` imports safe in tools files:

In [ ]:
toolsfile = Path(tempfile.mkdtemp())/'weather.py'
toolsfile.write_text(""""Answer weather questions."
from json import dumps

def _secret(): pass

def celsius(
    fahrenheit:float, # Temperature to convert
)->float:
    "Convert Fahrenheit to Celsius"
    return (fahrenheit-32)*5/9
""")
wsrv = load_server(toolsfile)
test_eq((wsrv.name, list(wsrv.funcs), wsrv.instructions), ('weather', ['celsius'], 'Answer weather questions.'))
wsrv.specs

[{'name': 'celsius',
  'description': 'Convert Fahrenheit to Celsius\n\nReturns:\n- type: number',
  'inputSchema': {'type': 'object',
   'properties': {'fahrenheit': {'description': 'Temperature to convert',
     'type': 'number'}},
   'required': ['fahrenheit']}}]

`main` is the console entry point, `call_parse` turning its docments into the CLI. The flags follow the shape MCP deployments already use (`--transport`, `--host`, `--port`, with the token from `$MCPMINI_TOKEN` and the `--no_token` opt-out); `stdio` is the default, so the common host launch line is just `mcpmini tools.py`:

In [ ]:
#| export
@call_parse
def main(
    tools:str, # Python file whose public docmented functions are served
    transport:str='stdio', # 'stdio', or 'http' for streamable HTTP
    host:str='127.0.0.1', # Interface to bind, for http
    port:int=8000, # Port to bind, for http
    token:str=None, # Bearer token, for http; `$MCPMINI_TOKEN` if unset
    no_token:bool=False, # Serve http on a non-loopback interface openly, when auth lives elsewhere
    path:str='/mcp', # Endpoint path, for http
):
    "Serve a Python file's docmented functions as an MCP server"
    srv = load_server(tools)
    if transport=='stdio': asyncio.run(serve_stdio(srv))
    elif transport in ('http','streamable-http'): asyncio.run(serve_mcp(srv, host=host, port=port, token=token, no_token=no_token, path=path))
    else: raise ValueError(f'unknown transport: {transport}')

End to end the way a host runs it: the installed `mcpmini` script, the tools file as its argument, our client on the other end of the pipe. The negotiated instructions arrive with the handshake, and the served tool converts:

In [ ]:
cmd = shutil.which('mcpmini')
assert cmd, 'mcpmini console script not installed: run `uv sync`'
async with MCPClient.stdio([cmd, str(toolsfile)]) as c:
    test_eq(c.info['instructions'], 'Answer weather questions.')
    freezing = await c.tools.celsius(fahrenheit=32)
freezing


'0.0'

## A live client

The interop tests above prove SDK compatibility; the proof that matters for deployment is a real MCP *host* wiring the server to a model. The Agent SDK runs Claude Code's own client machinery headless, so this is the genuine article: Claude launches the installed `mcpmini` script itself, discovers the tool, and must actually call it (the assertion checks the `tool_use`, not just the answer, since a model could convert temperatures unaided). These cells spend tokens, so they stay out of automated runs:

In [ ]:
#| eval: false
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage

In [ ]:
#| eval: false
logging.getLogger('claude_agent_sdk').setLevel(logging.WARNING)
live_tools = Path(tempfile.mkdtemp())/'weather.py'
live_tools.write_text(toolsfile.read_text())
opts = ClaudeAgentOptions(model='haiku', max_turns=3, tools=[], setting_sources=[], strict_mcp_config=True,
    mcp_servers=dict(demo=dict(type='stdio', command=shutil.which('mcpmini'), args=[str(live_tools)])),
    allowed_tools=['mcp__demo__celsius'], system_prompt='Use the tools you are given.')
msgs = [m async for m in query(prompt='Convert 212 fahrenheit to celsius with the celsius tool. Reply with only the number.', options=opts)]
tus = [b.name for m in msgs if isinstance(m, AssistantMessage) for b in m.content if getattr(b,'name',None)]
res = first(m.result for m in msgs if isinstance(m, ResultMessage))
test_eq(tus, ['mcp__demo__celsius'])
assert '100' in res
res

'100.0'

The same run over streamable HTTP with a bearer token is the remote-deployment shape — the server could be on another machine entirely, and this config (URL plus `Authorization` header) is exactly what `claude mcp add --transport http` records:

In [ ]:
#| eval: false
live_port = free_port()
live_task = asyncio.create_task(serve_mcp(wsrv, port=live_port, token='sesame'))
await asyncio.sleep(0.2)
hopts = ClaudeAgentOptions(model='haiku', max_turns=3, tools=[], setting_sources=[], strict_mcp_config=True,
    mcp_servers=dict(demo=dict(type='http', url=f'http://127.0.0.1:{live_port}/mcp', headers={'Authorization': 'Bearer sesame'})),
    allowed_tools=['mcp__demo__celsius'], system_prompt='Use the tools you are given.')
hmsgs = [m async for m in query(prompt='Convert 451 fahrenheit to celsius with the celsius tool. Reply with only the number.', options=hopts)]
htus = [b.name for m in hmsgs if isinstance(m, AssistantMessage) for b in m.content if getattr(b,'name',None)]
hres = first(m.result for m in hmsgs if isinstance(m, ResultMessage))
live_task.cancel()
test_eq(htus, ['mcp__demo__celsius'])
assert '232' in hres
hres

'232.78'

## Cleanup

Stop the two background servers.

In [ ]:
for t in (server_task, sdk_task): t.cancel()
res = await asyncio.gather(server_task, sdk_task, return_exceptions=True)


## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()
